# 01 Data Audit

Audit raw inputs and manifests before any preprocessing. This notebook is read-only with respect to raw climate files: it checks what has already downloaded, but it does not unzip, regrid, aggregate, or run spatial simulations.

## Step 14 Boundary

- Audit the expanded full-country AgERA5 request in `01_inputs/raw/agera5_full_extent`.
- Check yearly-variable coverage, ZIP CRC integrity, daily NetCDF member counts, and daily date coverage.
- Do not start CMIP6 downloads here.
- Do not create processed rasters here.
- Do not extract, regrid, aggregate, or otherwise process raw climate data in this audit.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
from zipfile import BadZipFile, ZipFile

import pandas as pd

RAW_AGERA5 = Path('../01_inputs/raw/agera5_full_extent')
EXPECTED_VARIABLES = [
    'precipitation',
    'reference_et',
    'solar_radiation',
    'tmax',
    'tmean',
    'tmin',
    'vpd',
]
EXPECTED_YEARS = range(2000, 2021)

RAW_AGERA5

In [ ]:
def parse_agera5_name(path: Path) -> tuple[str | None, int | None]:
    stem = path.stem
    if not stem.startswith('agera5_'):
        return None, None
    prefix, year_text = stem.rsplit('_', 1)
    variable = prefix.removeprefix('agera5_')
    try:
        year = int(year_text)
    except ValueError:
        return variable, None
    return variable, year


def zip_status(path: Path) -> str:
    if not path.exists() or path.stat().st_size == 0:
        return 'missing_or_empty'
    try:
        with ZipFile(path) as archive:
            bad_member = archive.testzip()
    except BadZipFile:
        return 'bad_zip'
    return 'ok' if bad_member is None else f'bad_member:{bad_member}'

In [ ]:
records = []
for path in sorted(RAW_AGERA5.glob('agera5_*.zip')):
    variable, year = parse_agera5_name(path)
    records.append(
        {
            'path': str(path),
            'variable': variable,
            'year': year,
            'size_mb': path.stat().st_size / 1_000_000,
            'zip_status': zip_status(path),
        }
    )

inventory = pd.DataFrame.from_records(records)
inventory

In [ ]:
if inventory.empty:
    coverage = pd.DataFrame(columns=['variable', 'downloaded_years', 'ok_archives'])
else:
    coverage = (
        inventory.groupby('variable', dropna=False)
        .agg(downloaded_years=('year', 'nunique'), ok_archives=('zip_status', lambda values: sum(value == 'ok' for value in values)))
        .reset_index()
    )

coverage

In [ ]:
downloaded = set(zip(inventory.get('variable', []), inventory.get('year', [])))
expected = {(variable, year) for variable in EXPECTED_VARIABLES for year in EXPECTED_YEARS}
missing = sorted(expected - downloaded)
invalid = inventory[inventory.get('zip_status', pd.Series(dtype=str)) != 'ok'] if not inventory.empty else inventory

summary = {
    'expected_archives': len(expected),
    'downloaded_archives': len(downloaded & expected),
    'missing_archives': len(missing),
    'invalid_archives': len(invalid),
    'status': 'complete' if not missing and len(invalid) == 0 else 'in_progress_or_needs_attention',
}
summary

In [ ]:
import calendar
import re

daily_checks = []
for row in inventory.itertuples(index=False):
    if row.zip_status != 'ok' or row.variable not in EXPECTED_VARIABLES or row.year not in EXPECTED_YEARS:
        continue
    with ZipFile(row.path) as archive:
        members = [name for name in archive.namelist() if not name.endswith('/')]
    netcdf_members = [name for name in members if name.lower().endswith(('.nc', '.nc4'))]
    expected_days = 366 if calendar.isleap(row.year) else 365
    observed_dates = []
    for name in netcdf_members:
        match = re.search(r'_(\d{8})_', Path(name).name)
        if match:
            observed_dates.append(match.group(1))
    expected_dates = {
        f'{row.year}{month:02d}{day:02d}'
        for month in range(1, 13)
        for day in range(1, calendar.monthrange(row.year, month)[1] + 1)
    }
    date_counts = Counter(observed_dates)
    daily_checks.append(
        {
            'variable': row.variable,
            'year': row.year,
            'member_count_ok': len(members) == expected_days and len(netcdf_members) == expected_days,
            'date_coverage_ok': set(observed_dates) == expected_dates and all(count == 1 for count in date_counts.values()),
            'missing_dates': len(expected_dates - set(observed_dates)),
            'duplicate_dates': sum(count > 1 for count in date_counts.values()),
        }
    )

daily_coverage = pd.DataFrame.from_records(daily_checks)
daily_coverage_issues = daily_coverage[
    ~(daily_coverage['member_count_ok'] & daily_coverage['date_coverage_ok'])
]
audit_status = {
    'daily_archives_checked': len(daily_coverage),
    'daily_coverage_issues': len(daily_coverage_issues),
    'status': 'complete' if summary['status'] == 'complete' and len(daily_coverage) == len(expected) and daily_coverage_issues.empty else 'in_progress_or_needs_attention',
}
daily_coverage, audit_status

In [ ]:
Counter(variable for variable, _year in missing), invalid